# NB02 — Load Testing: Throughput & Concurrency

Measures **requests-per-second (RPS)**, **error rate under load**, and plots the
**latency-vs-concurrency curve** for the converter endpoint.

**Approach:** Pure asyncio + httpx (no separate locust process required).  
**Requires:** Backend running at `http://localhost:8000`  
**Output:** `../results/load/rps.csv`, `../results/load/concurrency_curve.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'scripts'))

import asyncio
import time
import statistics
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

import httpx
from api_timing import (
    get_auth_token, auth_headers,
    concurrent_burst, run_concurrency_sweep,
)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})

RESULTS_DIR = Path('../results/load')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = 'http://localhost:8000'
print('✓ Imports OK')

In [ ]:
token = get_auth_token()
hdrs = auth_headers(token)
print(f'Auth token: {"✓" if token else "⚠ none"}')

## 1 — Sustained Throughput (Health Endpoint Baseline)

We use the `/api/health` endpoint as a baseline since it has zero compute cost.
This gives us the upper bound RPS that the server framework can serve.

In [ ]:
async def sustained_rps(url: str, duration_s: float = 10.0, headers: dict | None = None) -> dict:
    """Fire as many requests as possible in duration_s seconds."""
    results = []
    deadline = time.perf_counter() + duration_s
    async with httpx.AsyncClient(timeout=10) as client:
        while time.perf_counter() < deadline:
            t0 = time.perf_counter()
            try:
                r = await client.get(url, headers=headers or {})
                results.append({'ok': r.status_code < 400, 'ms': (time.perf_counter() - t0) * 1000})
            except Exception:
                results.append({'ok': False, 'ms': (time.perf_counter() - t0) * 1000})
    latencies = [r['ms'] for r in results]
    return {
        'total_requests': len(results),
        'rps': len(results) / duration_s,
        'error_rate': sum(1 for r in results if not r['ok']) / len(results),
        'mean_ms': statistics.mean(latencies),
        'p95_ms': sorted(latencies)[int(0.95 * len(latencies))],
    }

print('Running 10s sustained load on /api/health...')
health_rps = asyncio.run(sustained_rps(f'{BASE_URL}/api/health', duration_s=10, headers=hdrs))
print(f"RPS: {health_rps['rps']:.1f}  p95: {health_rps['p95_ms']:.1f}ms  errors: {health_rps['error_rate']:.1%}")

## 2 — Concurrency Sweep (Converter Endpoint)

Ramp up concurrent users from 1 to 50 and measure how latency and throughput change.

In [ ]:
CONVERT_URL = f'{BASE_URL}/api/converter/convert'
CIRQ_PAYLOAD = {
    'code': (
        'import cirq\n'
        'q0, q1 = cirq.LineQubit.range(2)\n'
        'circuit = cirq.Circuit([cirq.H(q0), cirq.CNOT(q0, q1), cirq.measure(q0, q1, key="r")])\n'
    ),
    'framework': 'cirq',
}

CONCURRENCY_LEVELS = [1, 2, 5, 10, 20, 30, 50]
print('Running concurrency sweep on /api/converter/convert...')
df_sweep = run_concurrency_sweep(
    'POST', CONVERT_URL,
    levels=CONCURRENCY_LEVELS,
    headers=hdrs,
    json=CIRQ_PAYLOAD,
)
df_sweep.to_csv(RESULTS_DIR / 'concurrency_curve.csv', index=False)
print('\n✓ Saved concurrency_curve.csv')
df_sweep

## 3 — Plot: Latency vs. Concurrency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_sweep['concurrency'], df_sweep['mean_ms'], 'o-', color='steelblue', label='Mean', linewidth=2)
axes[0].plot(df_sweep['concurrency'], df_sweep['p95_ms'], 's--', color='tomato', label='p95', linewidth=2)
axes[0].fill_between(df_sweep['concurrency'], df_sweep['mean_ms'], df_sweep['p95_ms'], alpha=0.15, color='steelblue')
axes[0].axhline(2000, color='red', linestyle='--', linewidth=1, label='2s target')
axes[0].set_xlabel('Concurrent Users')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Latency vs Concurrency (Converter)')
axes[0].legend()

axes[1].plot(df_sweep['concurrency'], df_sweep['throughput_rps'], 'D-', color='seagreen', linewidth=2)
axes[1].set_xlabel('Concurrent Users')
axes[1].set_ylabel('Throughput (RPS)')
axes[1].set_title('Throughput vs Concurrency')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'concurrency_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved concurrency_curve.png')

## 4 — Error Rate Under Load

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(df_sweep['concurrency'].astype(str), df_sweep['error_rate'] * 100, color='orchid')
ax.set_xlabel('Concurrent Users')
ax.set_ylabel('Error Rate (%)')
ax.set_title('Error Rate vs Concurrency')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1f}%'))
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.2, f'{h:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'error_rate.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved error_rate.png')

## 5 — Multi-Endpoint RPS Summary

Compare peak RPS across endpoint types (health, converter, simulator).

In [ ]:
SIM_PAYLOAD = {
    'qasm_code': 'OPENQASM 3.0;\nqubit[2] q;\nbit[2] c;\nh q[0];\ncx q[0], q[1];\nc[0] = measure q[0];\nc[1] = measure q[1];\n',
    'backend': 'statevector',
    'shots': 256,
}

FIXED_CONCURRENCY = 10  # representative user count
rps_summary = []

endpoints = [
    ('GET /api/health', 'GET', f'{BASE_URL}/api/health', None),
    ('POST /api/converter/convert', 'POST', CONVERT_URL, CIRQ_PAYLOAD),
    ('POST /api/simulator/simulate', 'POST', f'{BASE_URL}/api/simulator/simulate', SIM_PAYLOAD),
]

for label, method, url, payload in endpoints:
    result = asyncio.run(concurrent_burst(method, url, n_concurrent=FIXED_CONCURRENCY, headers=hdrs, json=payload))
    result['endpoint'] = label
    rps_summary.append(result)
    print(f"{label}: RPS={result['throughput_rps']:.1f}  p95={result['p95_ms']:.0f}ms  err={result['error_rate']:.1%}")

df_rps = pd.DataFrame(rps_summary)
df_rps.to_csv(RESULTS_DIR / 'rps.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(df_rps['endpoint'], df_rps['throughput_rps'], color=['seagreen', 'steelblue', 'orchid'])
ax.set_xlabel('Throughput (RPS at 10 concurrent users)')
ax.set_title('Peak Throughput per Endpoint Type')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'rps.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved rps.csv + rps.png')

## 6 — Locust Command Reference

For a GUI-based load test with detailed statistics, run locust standalone:

In [ ]:
import sys, subprocess
scripts_dir = os.path.join(os.path.dirname(os.getcwd()), 'scripts')
cmd = (
    f'locust -f {scripts_dir}/locust_scenarios.py '
    f'--headless --host {BASE_URL} -u 20 -r 5 -t 60s '
    f'--only-summary --csv {RESULTS_DIR}/locust'
)
print('Run this command to execute a 60-second locust load test:')
print(f'\n  {cmd}\n')
print('Or visit http://localhost:8089 for the web UI if you run locust without --headless.')